# Train the Distant Contradiction Auditor (Colab T4)

**Runtime → Change runtime type → GPU → T4.**

A long contract is **not** fed into BERT as one sequence (512-token limit). The trained piece is a **pair classifier**. At inference:

`long document → segment clauses → propose pairs (next / cross-ref / shared term / far) → Legal-BERT scores each pair → JSON [{span_a, span_b, type, score}]`

### Upload to Drive `MyDrive/dca/` (or use the file picker below)

| File | Required |
|---|---|
| `pairs_colab.jsonl.gz` (or `pairs.jsonl`) | **Yes** — training |
| `llm_testset_long.jsonl` | Optional — document-level metrics on ≥20-page CUAD test contracts |

Do **not** train on NVIDIA outputs. LLM comparison is eval-only (last optional cell).

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn

import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — switch to T4')

from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print('Drive mount skipped:', e)

In [ ]:
from pathlib import Path
import gzip, shutil, json

DRIVE = Path('/content/drive/MyDrive/dca')
NESTED = Path('/content/drive/MyDrive/MyDrive/dca')
LOCAL = Path('/content')

def _unzip(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(src, 'rb') as inf, open(dst, 'wb') as out:
        shutil.copyfileobj(inf, out)

def _first_existing(paths):
    return next((p for p in paths if p.exists()), None)

PAIRS = _first_existing([
    DRIVE / 'pairs_colab.jsonl', NESTED / 'pairs_colab.jsonl',
    DRIVE / 'pairs.jsonl', NESTED / 'pairs.jsonl',
    LOCAL / 'pairs_colab.jsonl', LOCAL / 'pairs.jsonl',
])
gz = _first_existing([
    DRIVE / 'pairs_colab.jsonl.gz', NESTED / 'pairs_colab.jsonl.gz',
    LOCAL / 'pairs_colab.jsonl.gz',
])
if PAIRS is None and gz is not None:
    dest_dir = gz.parent if gz.parent != Path('.') else LOCAL
    PAIRS = dest_dir / 'pairs_colab.jsonl'
    _unzip(gz, PAIRS)

if PAIRS is None or not PAIRS.exists():
    from google.colab import files
    print('Upload artifacts/pairs_colab.jsonl.gz from the laptop')
    files.upload()
    gz = LOCAL / 'pairs_colab.jsonl.gz'
    PAIRS = LOCAL / 'pairs_colab.jsonl'
    if gz.exists():
        _unzip(gz, PAIRS)

assert PAIRS is not None and PAIRS.exists(), 'Need pairs_colab.jsonl.gz'
OUT = (DRIVE if DRIVE.exists() else NESTED if NESTED.exists() else LOCAL) / 'legalbert-dca'
docs_path = None
docs = []
print('pairs', PAIRS, 'bytes', PAIRS.stat().st_size)
print('OUT', OUT)

## Train Legal-BERT as a pair classifier

This is the only learned component. Input: `(text_a, text_b)` from BCC. Label: conflict vs not. 2 epochs, fp16, batch 8, accum 4, max_length 256.

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, f1_score

MODEL = 'nlpaueb/legal-bert-base-uncased'
MAX_LEN = 256

rows = [json.loads(l) for l in PAIRS.read_text().splitlines() if l.strip()]
by = {}
for r in rows:
    by.setdefault(r['split'], []).append({
        'text_a': r['text_a'][:1200],
        'text_b': r['text_b'][:1200],
        'label': int(r['label']),
        'gap_bin': r.get('gap_bin', 'unknown'),
        'type': r.get('type', 'Inconsistency'),
    })
ds = DatasetDict({k: Dataset.from_list(v) for k, v in by.items()})
print(ds)
assert 'train' in ds, 'pairs file must have split=train rows'

tok = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tok(batch['text_a'], batch['text_b'], truncation=True, max_length=MAX_LEN, padding=False)

keep = ('label', 'gap_bin', 'type')
enc = ds.map(tokenize, batched=True, remove_columns=[c for c in ds['train'].column_names if c not in keep])
enc = enc.rename_column('label', 'labels')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    return {'precision': p, 'recall': r, 'f1': f1}

args = TrainingArguments(
    output_dir='/content/dca-out',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    report_to=[],
    seed=13,
)
eval_name = 'dev' if 'dev' in enc else 'test'
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=enc['train'],
    eval_dataset=enc[eval_name],
    processing_class=tok,
    compute_metrics=compute_metrics,
)
trainer.train()
OUT.mkdir(parents=True, exist_ok=True)
trainer.save_model(OUT)
tok.save_pretrained(OUT)
print('saved', OUT)

In [ ]:
pred = trainer.predict(enc['test'])
logits, labels = pred.predictions, pred.label_ids
preds = np.argmax(logits, axis=-1)
p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
test_rows = by['test']
by_bin = {}
for i, row in enumerate(test_rows):
    if row['label'] != 1:
        continue
    b = row['gap_bin']
    by_bin.setdefault(b, {'yt': [], 'yp': []})
    by_bin[b]['yt'].append(1)
    by_bin[b]['yp'].append(int(preds[i]))
gap = {k: float(f1_score(v['yt'], v['yp'], zero_division=0)) for k, v in by_bin.items()}
pair_metrics = {
    'model': MODEL,
    'test_precision': float(p),
    'test_recall': float(r),
    'test_f1': float(f1),
    'f1_by_gap_bin_gold_positives': gap,
    'n_test': len(test_rows),
}
metrics_path = OUT.parent / 'metrics_legalbert.json'
metrics_path.write_text(json.dumps(pair_metrics, indent=2))
print(json.dumps(pair_metrics, indent=2))
print('wrote', metrics_path)

## Optional: long-document reviewer metrics

Upload `llm_testset_long.jsonl` to the same Drive folder (full CUAD text, frozen test ids, ≥20 pages). This scores the **full pipeline**, not pair-only F1. Skip if you do not have that file yet.

In [ ]:
import re
from dataclasses import dataclass
from collections import defaultdict

HEADING_RE = re.compile(r'(?m)^(?P<head>\s*(?:Section\s+)?\d+(?:\.\d+)*\s*(?:\([a-z0-9]+\))?\.?\s+[A-Z][^\n]{0,80})')
ALT_HEADING_RE = re.compile(r'(?m)^(?P<head>\s*(?:ARTICLE|Article|SECTION|Section)\s+[IVXLC\d]+[^\n]{0,80})')
REF_RE = re.compile(r'(?:Section|Clause|Article)\s+(\d+(?:\.\d+)*)', re.I)
TERM_RE = re.compile(r'\b([A-Z][A-Za-z]{3,}(?:\s+[A-Z][A-Za-z]{3,}){0,3})\b')

@dataclass
class Clause:
    clause_id: str
    heading: str
    text: str
    start: int
    end: int
    order: int

def segment_contract(text: str):
    text = text.replace('\r\n', '\n')
    matches = []
    for rx in (HEADING_RE, ALT_HEADING_RE):
        for m in rx.finditer(text):
            matches.append((m.start(), m.group('head').strip()))
    uniq, seen = [], set()
    for start, head in sorted(set(matches)):
        if start in seen:
            continue
        seen.add(start)
        uniq.append((start, head))
    clauses = []
    if len(uniq) >= 3:
        for i, (start, head) in enumerate(uniq):
            end = uniq[i + 1][0] if i + 1 < len(uniq) else len(text)
            body = text[start:end].strip()
            if len(body) < 40:
                continue
            clauses.append(Clause(f'c{len(clauses)}', head[:120], body[:4000], start, end, len(clauses)))
        if clauses:
            return clauses
    paras = [p.strip() for p in re.split(r'\n\s*\n', text) if len(p.strip()) > 80] or [text.strip()[:4000]]
    for i, para in enumerate(paras[:80]):
        start = text.find(para[:80]) if para else 0
        clauses.append(Clause(f'c{i}', para.split('\n', 1)[0][:80], para[:4000], max(start, 0), max(start, 0) + len(para), i))
    return clauses

def candidate_pairs(clauses, max_pairs=200):
    n = len(clauses)
    scored = {}
    def add(i, j, reason):
        if i == j:
            return
        a, b = (i, j) if i < j else (j, i)
        scored.setdefault((a, b), reason)
    for i in range(n - 1):
        add(i, i + 1, 'NEXT')
        if i + 2 < n:
            add(i, i + 2, 'NEXT')
    heading_index = {}
    for i, c in enumerate(clauses):
        m = re.search(r'(\d+(?:\.\d+)*)', c.heading)
        if m:
            heading_index[m.group(1)] = i
    for i, c in enumerate(clauses):
        for ref in REF_RE.findall(c.text[:1500]):
            if ref in heading_index:
                add(i, heading_index[ref], 'REFERENCES')
    term_map = {}
    for i, c in enumerate(clauses):
        for t in list({x.lower() for x in TERM_RE.findall(c.text[:800]) if len(x) > 5})[:12]:
            term_map.setdefault(t, []).append(i)
    for ids in term_map.values():
        u = sorted(set(ids))
        if 2 <= len(u) <= 6:
            add(u[0], u[-1], 'SHARED_TERM')
    if n >= 6:
        for i in range(min(12, max(4, n // 4))):
            add(i, n - 1 - i, 'FAR')
    pairs = [(a, b, r) for (a, b), r in scored.items()]
    pairs.sort()
    return pairs[:max_pairs]

def tokens(text):
    return {t for t in re.findall(r'[a-z0-9]+', (text or '').lower()) if len(t) > 1}

def jaccard(a, b):
    ta, tb = tokens(a), tokens(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)

def pair_sim(pred, gold, thr=0.5):
    sa, sb, ga, gb = pred.get('span_a',''), pred.get('span_b',''), gold.get('span_a',''), gold.get('span_b','')
    score = max(min(jaccard(sa, ga), jaccard(sb, gb)), min(jaccard(sa, gb), jaccard(sb, ga)))
    return score if score >= thr else 0.0

def document_prf(predicted, gold, thr=0.5):
    scored = []
    for i, pred in enumerate(predicted):
        for j, g in enumerate(gold):
            s = pair_sim(pred, g, thr)
            if s > 0:
                scored.append((s, i, j))
    scored.sort(reverse=True)
    used_p, used_g = set(), set()
    for _, i, j in scored:
        if i in used_p or j in used_g:
            continue
        used_p.add(i); used_g.add(j)
    tp, fp, fn = len(used_p), len(predicted) - len(used_p), len(gold) - len(used_g)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    gap = defaultdict(lambda: {'tp': 0, 'fn': 0})
    for j, g in enumerate(gold):
        b = g.get('gap_bin') or 'unknown'
        if j in used_g:
            gap[b]['tp'] += 1
        else:
            gap[b]['fn'] += 1
    recall_by_gap = {k: v['tp'] / (v['tp'] + v['fn']) if (v['tp'] + v['fn']) else 0.0 for k, v in gap.items()}
    return {'tp': tp, 'fp': fp, 'fn': fn, 'precision': prec, 'recall': rec, 'f1': f1, 'n_pred': len(predicted), 'n_gold': len(gold), 'recall_by_gap': recall_by_gap, 'gap_counts': dict(gap)}

print('pipeline helpers ready')

In [ ]:
docs_path = _first_existing([
    DRIVE / 'llm_testset_long.jsonl', NESTED / 'llm_testset_long.jsonl',
    LOCAL / 'llm_testset_long.jsonl',
])
if docs_path is None:
    print('No llm_testset_long.jsonl — pair metrics above are the training result. Upload the long-doc file to Drive/dca to run this cell.')
else:
    docs = [json.loads(l) for l in docs_path.read_text().splitlines() if l.strip()]
    print('docs', len(docs), docs_path)
    model.eval()
    per = []
    for di, doc in enumerate(docs):
        clauses = segment_contract(doc['text'])
        pairs = candidate_pairs(clauses, max_pairs=200)
        findings = []
        for i, j, reason in pairs:
            a, b = clauses[i], clauses[j]
            enc_p = tok(a.text[:1200], b.text[:1200], truncation=True, max_length=256, return_tensors='pt')
            enc_p = {k: v.to(model.device) for k, v in enc_p.items()}
            with torch.no_grad():
                logits_p = model(**enc_p).logits[0]
                proba = float(torch.softmax(logits_p, dim=-1)[1])
            if proba < 0.55:
                continue
            findings.append({'span_a': a.text[:900], 'span_b': b.text[:900], 'score': proba, 'type': 'Inconsistency'})
        findings.sort(key=lambda x: -x['score'])
        stats = document_prf(findings[:40], doc.get('gold') or [], thr=0.5)
        stats['contract_id'] = doc.get('contract_id')
        per.append(stats)
        print(di + 1, '/', len(docs), 'F1', round(stats['f1'], 3), 'P', round(stats['precision'], 3), 'R', round(stats['recall'], 3))
    tp, fp, fn = sum(d['tp'] for d in per), sum(d['fp'] for d in per), sum(d['fn'] for d in per)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1d = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    doc_metrics = {'n_docs': len(per), 'precision': prec, 'recall': rec, 'f1': f1d, 'tp': tp, 'fp': fp, 'fn': fn}
    (OUT.parent / 'metrics_longdocs.json').write_text(json.dumps(doc_metrics, indent=2))
    print(json.dumps(doc_metrics, indent=2))

## Optional: NVIDIA eval (do not train on this)

In Colab: `os.environ['NVIDIA_API_KEY'] = '…'` then run. Same Jaccard matching. Skip if you have no key.

In [ ]:
import os, urllib.request

NVIDIA_MODEL = os.environ.get('NVIDIA_MODEL', 'openai/gpt-oss-120b')
key = os.environ.get('NVIDIA_API_KEY') or os.environ.get('NGC_API_KEY')
if not key:
    print('Set NVIDIA_API_KEY to compare gpt-oss on the same long docs.')
elif docs_path is None:
    print('Need llm_testset_long.jsonl first.')
else:
    sys_msg = (
        'You are a contract reviewer. List disagreeing clause pairs. '
        'JSON only: [{span_a, span_b, type}]. Types: Ambiguity, Inconsistency, '
        'MisalignedTerminology, Omission, StructuralFlaws. Prefer precision.'
    )
    llm_per = []
    for doc in docs:
        body = {
            'model': NVIDIA_MODEL,
            'temperature': 0.2,
            'messages': [
                {'role': 'system', 'content': sys_msg},
                {'role': 'user', 'content': 'Contract:\n' + doc['text'][:120000] + '\n\nJSON list:'},
            ],
        }
        req = urllib.request.Request(
            'https://integrate.api.nvidia.com/v1/chat/completions',
            data=json.dumps(body).encode(),
            headers={'Authorization': f'Bearer {key}', 'Content-Type': 'application/json'},
        )
        try:
            with urllib.request.urlopen(req, timeout=180) as resp:
                content = json.loads(resp.read().decode())['choices'][0]['message']['content']
            m = re.search(r'\[.*\]', content, re.S)
            findings = json.loads(m.group(0) if m else content)
            if not isinstance(findings, list):
                findings = []
        except Exception as e:
            print('NVIDIA error', e)
            findings = []
        llm_per.append(document_prf(findings, doc.get('gold') or [], thr=0.5))
    tp, fp, fn = sum(d['tp'] for d in llm_per), sum(d['fp'] for d in llm_per), sum(d['fn'] for d in llm_per)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1d = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    print(json.dumps({'model': NVIDIA_MODEL, 'precision': prec, 'recall': rec, 'f1': f1d, 'n_docs': len(llm_per)}, indent=2))